In [ ]:
# ! pip install gensim

## import libraries

In [1]:
import pandas as pd
import re
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.pipeline import Pipeline
from gensim.models import Word2Vec
from sklearn.base import BaseEstimator, TransformerMixin

import warnings
warnings.filterwarnings("ignore")

/Users/negin/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## load datasets

In [2]:
train_df = pd.read_csv('data/6/train.csv')
valid_df = pd.read_csv('data/6/valid.csv')
# test_df = pd.read_csv('data/6/test.csv')

train_df.dropna(subset=['text', 'sentiment'], inplace=True)
valid_df.dropna(subset=['text', 'sentiment'], inplace=True)

## **Model Development** using Train set

- Vecotorizing (Preprocessing + Tokenization)

- Hyperparamete tuning using CV on Train set

- Re-train best model on Train set

In [6]:
results_df = pd.DataFrame(columns=["vectorizer", "valid_accuracy", "valid_F1_score", "grid_search"])

def run_pipeline(vectorizer_name, grid_search, train_df, valid_df, results_df=results_df):

    # Fit the model
    grid_search.fit(train_df['text'], train_df['sentiment'])

    # Get the best pipeline and make predictions
    best_pipeline = grid_search.best_estimator_
    train_predictions = best_pipeline.predict(train_df['text'])
    valid_predictions = best_pipeline.predict(valid_df['text'])

    # Best parameters
    print("Best parameters:")
    print(grid_search.best_params_)

    # Evaluation on train_df
    print("\nEvaluation on train_df:")
    print("Train Accuracy:", round(accuracy_score(train_df['sentiment'], train_predictions),2), '%')
    print("Train F1-macro:", round(f1_score(train_df['sentiment'], train_predictions, average='macro'),2), '%')
    print("\nClassification report on train_df:")
    print(classification_report(train_df['sentiment'], train_predictions))

    # Evaluation on valid_df
    print("\nEvaluation on valid_df:")
    print("Validation Accuracy:", round(accuracy_score(valid_df['sentiment'], valid_predictions),2), '%')
    print("Validation F1-macro:", round(f1_score(valid_df['sentiment'], valid_predictions, average='macro'),2), '%')
    print("\nClassification report on valid_df:")
    print(classification_report(valid_df['sentiment'], valid_predictions))

    # Append results to results_df
    results = {
        "vectorizer": vectorizer_name,
        "valid_accuracy": round(accuracy_score(valid_df['sentiment'], valid_predictions),2),
        "valid_F1_score": round(f1_score(valid_df['sentiment'], valid_predictions, average='macro'),2),
        "grid_search": grid_search
    }

    results_df = pd.concat([results_df, pd.DataFrame([results])], ignore_index=True)
    return results_df


### TF-IDF representation

Hyperparameter tuning using GridsearchCV

In [ ]:
tfidf_vectorizer = TfidfVectorizer(lowercase=True)
tfidf_model = LogisticRegression(max_iter=1000)

tfidf_pipeline = Pipeline([
    ('vectorizer', tfidf_vectorizer),
    ('classifier', tfidf_model)
])

tfidf_param_grid = {
    'vectorizer__ngram_range': [(1, 2)],
    'vectorizer__min_df': [2, 5, 10],
    'vectorizer__max_df': [0.9],
    'vectorizer__max_features': [5000, 10000],
    'vectorizer__sublinear_tf': [True],
    'classifier__C': [0.001, 0.01, 0.1, 1],
    'classifier__penalty': ['l2'],
    'classifier__solver': ['liblinear'],
    'classifier__class_weight': ['balanced']
}

tfidf_grid_search = GridSearchCV(
    estimator=tfidf_pipeline,
    param_grid=tfidf_param_grid,
    cv=5,
    scoring='f1_macro',
    n_jobs=-1
)

results_df = run_pipeline('TF-IDF', tfidf_grid_search, train_df, valid_df, results_df)
results_df

Best parameters:
{'classifier__C': 1, 'classifier__class_weight': 'balanced', 'classifier__penalty': 'l2', 'classifier__solver': 'liblinear', 'vectorizer__max_df': 0.9, 'vectorizer__max_features': 10000, 'vectorizer__min_df': 2, 'vectorizer__ngram_range': (1, 2), 'vectorizer__sublinear_tf': True}

Evaluation on train_df:
Train Accuracy: 0.92 %
Train F1-macro: 0.89 %

Classification report on train_df:
              precision    recall  f1-score   support

    negative       0.73      0.97      0.83      1001
    positive       0.99      0.91      0.95      3911

    accuracy                           0.92      4912
   macro avg       0.86      0.94      0.89      4912
weighted avg       0.94      0.92      0.92      4912


Evaluation on valid_df:
Validation Accuracy: 0.88 %
Validation F1-macro: 0.83 %

Classification report on valid_df:
              precision    recall  f1-score   support

    negative       0.65      0.85      0.74       143
    positive       0.96      0.88      0.9

,vectorizer,valid_accuracy,valid_F1_score,grid_search
0,TF-IDF,0.88,0.83,"GridSearchCV(cv=5,\n estimator=Pip..."


### BoW representation

In [8]:
bow_vectorizer = CountVectorizer(lowercase=True)
bow_model = LogisticRegression(max_iter=1000)

bow_pipeline = Pipeline([
    ('vectorizer', bow_vectorizer),
    ('classifier', bow_model)
])

bow_param_grid = {
    'vectorizer__ngram_range': [(1, 1), (1, 2)],
    'vectorizer__min_df': [2, 5, 10],
    'vectorizer__max_df': [0.9, 1.0],
    'vectorizer__max_features': [3000, 5000, 10000],
    'vectorizer__binary': [False, True],
    'classifier__C': [0.001, 0.01, 0.1, 1],
    'classifier__penalty': ['l2'],
    'classifier__solver': ['liblinear'],
    'classifier__class_weight': ['balanced']
}

bow_grid_search = GridSearchCV(
    estimator=bow_pipeline,
    param_grid=bow_param_grid,
    cv=5,
    scoring='f1_macro',
    n_jobs=-1
)

results_df = run_pipeline('BoW', bow_grid_search, train_df, valid_df, results_df)

Best parameters:
{'classifier__C': 0.1, 'classifier__class_weight': 'balanced', 'classifier__penalty': 'l2', 'classifier__solver': 'liblinear', 'vectorizer__binary': False, 'vectorizer__max_df': 0.9, 'vectorizer__max_features': 10000, 'vectorizer__min_df': 2, 'vectorizer__ngram_range': (1, 2)}

Evaluation on train_df:
Train Accuracy: 0.95 %
Train F1-macro: 0.93 %

Classification report on train_df:
              precision    recall  f1-score   support

    negative       0.84      0.95      0.89      1001
    positive       0.99      0.95      0.97      3911

    accuracy                           0.95      4912
   macro avg       0.92      0.95      0.93      4912
weighted avg       0.96      0.95      0.95      4912


Evaluation on valid_df:
Validation Accuracy: 0.9 %
Validation F1-macro: 0.84 %

Classification report on valid_df:
              precision    recall  f1-score   support

    negative       0.73      0.78      0.75       143
    positive       0.94      0.92      0.93   

### Word2vec representation

In [7]:
class Word2VecVectorizer(BaseEstimator, TransformerMixin):
    def __init__(self, vector_size=100, window=5, min_count=5, sg=0, workers=4):
        self.vector_size = vector_size
        self.window = window
        self.min_count = min_count
        self.sg = sg
        self.workers = workers
        self.w2v_model = None

    def fit(self, X, y=None):
        tokenized_texts = [self._tokenize(text) for text in X]
        self.w2v_model = Word2Vec(
            sentences=tokenized_texts,
            vector_size=self.vector_size,
            window=self.window,
            min_count=self.min_count,
            sg=self.sg,
            workers=self.workers
        )
        return self

    def transform(self, X):
        tokenized_texts = [self._tokenize(text) for text in X]
        return np.array([self._document_vector(tokens) for tokens in tokenized_texts])

    def _tokenize(self, text):
        return str(text).lower().split()

    def _document_vector(self, tokens):
        vectors = [self.w2v_model.wv[word] for word in tokens if word in self.w2v_model.wv]
        if len(vectors) == 0:
            return np.zeros(self.vector_size)
        return np.mean(vectors, axis=0)


w2v_vectorizer = Word2VecVectorizer()
w2v_model = LogisticRegression(max_iter=1000)

w2v_pipeline = Pipeline([
    ('vectorizer', w2v_vectorizer),
    ('classifier', w2v_model)
])

w2v_param_grid = {
    'vectorizer__vector_size': [50, 100],
    'vectorizer__window': [3, 5],
    'vectorizer__min_count': [5, 10],
    'vectorizer__sg': [0],
    'classifier__C': [0.001, 0.01, 0.1],
    'classifier__penalty': ['l2'],
    'classifier__solver': ['liblinear'],
    'classifier__class_weight': ['balanced']
}

w2v_grid_search = GridSearchCV(
    estimator=w2v_pipeline,
    param_grid=w2v_param_grid,
    cv=5,
    scoring='f1_macro',
    n_jobs=-1
)

results_df = run_pipeline('Word2Vec', w2v_grid_search, train_df, valid_df, results_df)

/Users/negin/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/negin/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/negin/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/negin/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with

Best parameters:
{'classifier__C': 0.001, 'classifier__class_weight': 'balanced', 'classifier__penalty': 'l2', 'classifier__solver': 'liblinear', 'vectorizer__min_count': 5, 'vectorizer__sg': 0, 'vectorizer__vector_size': 100, 'vectorizer__window': 5}

Evaluation on train_df:
Train Accuracy: 0.61 %
Train F1-macro: 0.58 %

Classification report on train_df:
              precision    recall  f1-score   support

    negative       0.33      0.83      0.47      1001
    positive       0.93      0.56      0.70      3911

    accuracy                           0.61      4912
   macro avg       0.63      0.70      0.58      4912
weighted avg       0.81      0.61      0.65      4912


Evaluation on valid_df:
Validation Accuracy: 0.62 %
Validation F1-macro: 0.59 %

Classification report on valid_df:
              precision    recall  f1-score   support

    negative       0.33      0.88      0.48       143
    positive       0.95      0.55      0.69       557

    accuracy                     

In [9]:
class TfidfWeightedWord2VecVectorizer(BaseEstimator, TransformerMixin):
    def __init__(self, vector_size=100, window=5, min_count=2, sg=1, workers=4):
        self.vector_size = vector_size
        self.window = window
        self.min_count = min_count
        self.sg = sg
        self.workers = workers
        self.w2v_model = None
        self.tfidf_vectorizer = None
        self.word2weight = {}

    def _tokenize(self, text):
        text = str(text).lower()
        return re.findall(r"\b\w+\b", text)

    def fit(self, X, y=None):
        tokenized_texts = [self._tokenize(text) for text in X]

        self.w2v_model = Word2Vec(
            sentences=tokenized_texts,
            vector_size=self.vector_size,
            window=self.window,
            min_count=self.min_count,
            sg=self.sg,
            workers=self.workers
        )

        joined_texts = [" ".join(tokens) for tokens in tokenized_texts]
        self.tfidf_vectorizer = TfidfVectorizer(tokenizer=str.split, lowercase=False)
        self.tfidf_vectorizer.fit(joined_texts)

        max_idf = max(self.tfidf_vectorizer.idf_)
        self.word2weight = {
            word: self.tfidf_vectorizer.idf_[idx]
            for word, idx in self.tfidf_vectorizer.vocabulary_.items()
        }
        self.max_idf = max_idf

        return self

    def transform(self, X):
        tokenized_texts = [self._tokenize(text) for text in X]
        return np.array([self._document_vector(tokens) for tokens in tokenized_texts])

    def _document_vector(self, tokens):
        weighted_vectors = []
        weights = []

        for word in tokens:
            if word in self.w2v_model.wv:
                weight = self.word2weight.get(word, self.max_idf)
                weighted_vectors.append(self.w2v_model.wv[word] * weight)
                weights.append(weight)

        if len(weighted_vectors) == 0:
            return np.zeros(self.vector_size)

        return np.sum(weighted_vectors, axis=0) / np.sum(weights)


w2v_vectorizer = TfidfWeightedWord2VecVectorizer()
w2v_model = LogisticRegression(max_iter=2000)

w2v_pipeline = Pipeline([
    ('vectorizer', w2v_vectorizer),
    ('classifier', w2v_model)
])

w2v_param_grid = {
    'vectorizer__vector_size': [100, 200],
    'vectorizer__window': [3, 5],
    'vectorizer__min_count': [2, 3],
    'vectorizer__sg': [0, 1],
    'classifier__C': [0.01, 0.1, 1],
    'classifier__penalty': ['l2'],
    'classifier__solver': ['liblinear'],
    'classifier__class_weight': ['balanced']
}

w2v_grid_search = GridSearchCV(
    estimator=w2v_pipeline,
    param_grid=w2v_param_grid,
    cv=5,
    scoring='f1_macro',
    n_jobs=1
)

results_df = run_pipeline('TF-IDF Weighted Word2Vec', w2v_grid_search, train_df, valid_df, results_df)

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_fl

Best parameters:
{'classifier__C': 1, 'classifier__class_weight': 'balanced', 'classifier__penalty': 'l2', 'classifier__solver': 'liblinear', 'vectorizer__min_count': 3, 'vectorizer__sg': 1, 'vectorizer__vector_size': 100, 'vectorizer__window': 5}

Evaluation on train_df:
Train Accuracy: 0.74 %
Train F1-macro: 0.7 %

Classification report on train_df:
              precision    recall  f1-score   support

    negative       0.43      0.88      0.58      1001
    positive       0.96      0.70      0.81      3911

    accuracy                           0.74      4912
   macro avg       0.70      0.79      0.70      4912
weighted avg       0.85      0.74      0.76      4912


Evaluation on valid_df:
Validation Accuracy: 0.73 %
Validation F1-macro: 0.68 %

Classification report on valid_df:
              precision    recall  f1-score   support

    negative       0.42      0.87      0.57       143
    positive       0.96      0.69      0.80       557

    accuracy                          

## **Model selection** using Validation set

In [29]:
best_model = results_df.sort_values(by='valid_F1_score', ascending=False).iloc[0]['vectorizer']
best_f1_score = results_df.sort_values(by='valid_F1_score', ascending=False).iloc[0]['valid_F1_score']

print('----------------- Model Performance Summary -----------------')
print(results_df.sort_values(by='valid_F1_score', ascending=False)[['vectorizer', 'valid_accuracy', 'valid_F1_score']])
print('\n------------------------ Final Model ------------------------')
print(f"Best model based on validation F1-score is {best_model} with F1-score of {round(best_f1_score * 100):.0f}%")

----------------- Model Performance Summary -----------------
                 vectorizer  valid_accuracy  valid_F1_score
1                       BoW            0.90            0.84
2  TF-IDF Weighted Word2Vec            0.73            0.68
0                  Word2Vec            0.62            0.59

------------------------ Final Model ------------------------
Best model based on validation F1-score is BoW with F1-score of 84%


## **Final Performance** using Test set